# 120 — Long-Context Agent
## What you will learn: full-document reasoning vs. chunked RAG
⏱ ~45 min

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/master/examples/120-long-context-agent/long_context_agent_workbook.ipynb)


## Part 1 — The Lost-in-the-Middle Problem

### Why context position matters

When you drop a long document into a model context window, you might assume
the model reads it uniformly. Research says otherwise.

Liu et al. (2023) — *"Lost in the Middle: How Language Models Use Long Contexts"*
([arxiv:2307.03172](https://arxiv.org/abs/2307.03172)) showed that:

- Models are **best** at using information near the **beginning** of context
- Models are **good** at using information near the **end** of context
- Models are **worst** at using information in the **middle** of context

```
Recall
  |****                                                    ****|
  |    **                                              **      |
  |      ***                                        ***        |
  |         ****                              *****             |
  |              **************************                     |
  +-------------------------------------------------------------->
  Start                   Position in doc                    End
```

**Practical implication:** critical information should be at the start or end of
your prompt, not buried in the middle.

This workbook demonstrates both strategies empirically on a 3,000-word synthetic report.


## Part 2 — Setup


In [ ]:
import sys

def _in_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False

if _in_colab():
    import subprocess
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "langchain-openai", "langchain-core", "python-dotenv"],
        check=True
    )
print("deps ready")


In [ ]:
import os

if _in_colab():
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
else:
    from dotenv import load_dotenv
    load_dotenv()

assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY in Colab Secrets or .env"
print("API key loaded")


## Part 3 — The Test Document

We use a synthetic ~3,000-word AI systems performance report.
It contains facts spread across 5 sections, ideal for testing multi-hop retrieval
(questions that require combining facts from different sections).


In [ ]:
LONG_DOCUMENT = """
# Annual AI Systems Performance Report 2024

## Executive Summary

This report covers enterprise AI deployments across 847 organizations surveyed in Q4 2024.
Organizations adopting multi-agent architectures achieved 34% higher task completion rates.
The average token cost per production query decreased by 41% year-over-year.

## Section 1: Deployment Patterns

### 1.1 Single-Agent vs. Multi-Agent

Among surveyed organizations, 62% use single-agent architectures for customer-facing applications,
while 38% have adopted multi-agent systems. In 2023, only 18% used multi-agent architectures.

Key drivers: complex task decomposition (78%), specialization (65%), parallel processing (54%),
reduced context window pressure per agent (47%).

### 1.2 Context Window Utilization

The "lost in the middle" phenomenon (Liu et al., 2023, arxiv:2307.03172) was observed in
42% of long-context deployments. When documents exceeded 32K tokens, models showed
an 18% decrease in accuracy for questions about content in the middle third of the document.

### 1.3 Model Selection Patterns

GPT-4o-mini: 44% of production queries (cost optimization)
GPT-4o: 31% of production queries (quality-critical tasks)
Claude Sonnet: 15% of production queries (long context + document tasks)
Other models: 10% (specialized or on-premise deployments)

## Section 2: Performance Metrics

### 2.1 Faithfulness and Hallucination Rates

| Approach | Faithfulness | Latency (p50) | Cost per 1K queries |
|----------|-------------|---------------|---------------------|
| Full document in context | 0.91 | 4.2s | $0.84 |
| Chunked RAG (top-3) | 0.83 | 1.8s | $0.22 |
| Chunked RAG (top-5) | 0.87 | 2.1s | $0.28 |
| No context (pure LLM) | 0.61 | 0.9s | $0.09 |

### 2.2 Multi-Hop Reasoning

Full-context approaches answered 89% of multi-hop questions correctly.
Chunked RAG answered 67% correctly -- a 25% gap.
The gap widens for 3+ hop questions: 82% vs 41%.

### 2.3 Latency Distribution

P50 latency (full context): 4.2 seconds. P95: 12.1 seconds.
P50 latency (chunked RAG): 1.8 seconds. P95: 4.3 seconds.
Time-to-first-token for full context is 2.3x slower than chunked RAG at p50.

## Section 3: Economic Analysis

### 3.1 Cost Per Accurate Answer

Full context cost per correct answer: $0.92
Chunked RAG cost per correct answer: $0.27 (at 83% faithfulness)
At scale (1M queries/month), the cost difference is $650K annually.

### 3.2 Break-Even Analysis

For tasks requiring >88% faithfulness, full-context approaches are cost-justified.
For tasks tolerating 80-88% faithfulness, chunked RAG is optimal.
For tasks tolerating <80% faithfulness, no-context LLM calls are sufficient.

### 3.3 Token Cost Trends

Year-over-year token cost reduction: 41%
Projected 2025 reduction: additional 35%
Organizations with dedicated prompt engineering teams: 29% lower costs on average.

## Section 4: Operational Findings

### 4.1 Chunk Size Optimization

Optimal chunk size for general document QA: 512-768 tokens.
Optimal overlap: 10-15% of chunk size (51-77 tokens for 512-token chunks).
Chunks below 256 tokens showed 23% higher hallucination rates.
Chunks above 1024 tokens showed 18% degraded retrieval precision.

### 4.2 Embedding Model Selection

text-embedding-3-small: 89% of chunked RAG deployments (cost-accuracy tradeoff)
text-embedding-3-large: 8% of deployments (highest accuracy)
Legacy ada-002: 3% of deployments (being phased out)

Retrieval accuracy (NDCG@5):
- text-embedding-3-small: 0.847
- text-embedding-3-large: 0.891
- ada-002: 0.801

## Section 5: Recommendations

### 5.1 Decision Framework

Use full-context when: document under 50K tokens, multi-hop reasoning required,
faithfulness >88% needed, latency tolerance >3 seconds.

Use chunked RAG when: corpus >100K tokens, fact-lookup queries, cost-sensitive,
latency <2 seconds required.

### 5.2 Implementation Checklist

For chunked RAG: chunk size 512-768 tokens, 10-15% overlap,
text-embedding-3-small as default, monitor NDCG@5 weekly.

For full-context: explicit length limits, critical info in first/last thirds,
use prompt caching for repeated document queries.

## Conclusion

By 2026, 60% of enterprise AI deployments will use full-context for documents
under 100K tokens, up from the current 23%.
"""

words = len(LONG_DOCUMENT.split())
print(f"Document stats: {words} words, {len(LONG_DOCUMENT)} characters")
print(f"Estimated tokens: ~{len(LONG_DOCUMENT) // 4}")


## Part 4 — Full-Context Approach

### How it works

The entire document is placed in the system prompt. The model has access
to all information simultaneously.

```
System prompt:
  You are a document analyst.
  DOCUMENT: [all 3,000 words here]

User: "What percentage use multi-agent AND what is faithfulness?"

Model: reads Section 1 + Section 2 simultaneously -> correct multi-hop answer
```

**When to use:** documents under 50K tokens, multi-hop questions, faithfulness > 88%

**Tradeoffs:** 4.2s p50 latency, $0.84/1K queries


In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

MODEL = "gpt-4o-mini"

def answer_with_full_context(document: str, questions: list) -> list:
    """Answer questions by loading the full document into context."""
    client = ChatOpenAI(model=MODEL, temperature=0)
    system = (
        "You are a precise document analyst. Answer questions based ONLY on the "
        "provided document. Be brief and cite specific numbers.\n\nDOCUMENT:\n"
        + document
    )
    answers = []
    for item in questions:
        msg = [SystemMessage(content=system), HumanMessage(content=item["q"])]
        response = client.invoke(msg)
        answers.append(response.content.strip())
    return answers

print("answer_with_full_context defined")


In [ ]:
QUESTIONS = [
    {"q": "What percentage of organizations use multi-agent architectures?",
     "expected": "38%"},
    {"q": "What is the faithfulness score for the full-document approach compared to chunked RAG top-3?",
     "expected": "0.91 vs 0.83"},
    {"q": "What is the optimal chunk size range for general document QA?",
     "expected": "512-768 tokens"},
    {"q": "How much did token costs decrease year-over-year?",
     "expected": "41%"},
    {"q": "What percentage cost reduction was achieved by organizations that use the most popular model (GPT-4o-mini at 44%) and also have dedicated prompt engineering teams?",
     "expected": "29% lower costs"},
]

print("5 questions loaded:")
for i, q in enumerate(QUESTIONS, 1):
    print(f"  Q{i}: {q['q'][:70]}...")


In [ ]:
import time

print("Running full-context approach (all 5 questions)...\n")
t0 = time.time()
full_answers = answer_with_full_context(LONG_DOCUMENT, QUESTIONS)
full_latency = time.time() - t0

for i, (q, a) in enumerate(zip(QUESTIONS, full_answers), 1):
    print(f"Q{i}: {q['q'][:60]}...")
    print(f"  Expected: {q['expected']}")
    print(f"  Got:      {a[:100]}")
    print()

print(f"Total latency: {full_latency:.1f}s ({full_latency/len(QUESTIONS):.1f}s per question)")


## Part 5 — Chunked RAG Approach

### How it works

1. **Chunk** the document into overlapping segments (~400 words)
2. **Retrieve** the top-K chunks most relevant to the query
3. **Inject** only those chunks into context

```
Document (3,000 words)
  chunk(size=400, overlap=50)
Chunks: [C1][C2][C3]...[C8]
  retrieve(query, top_k=3)
Context: [C2][C5][C7]  <- only relevant chunks
  LLM
Answer
```

**When to use:** large corpora (>100K tokens), fact-lookup queries, cost-sensitive

**Tradeoffs:** 1.8s p50 latency, $0.22/1K queries,
but 20-25% accuracy gap on multi-hop questions


In [ ]:
def _simple_chunk(text: str, chunk_size: int = 400, overlap: int = 50) -> list:
    """Split text into overlapping word-count chunks."""
    words = text.split()
    chunks = []
    i = 0
    while i < len(words):
        chunk = " ".join(words[i: i + chunk_size])
        chunks.append(chunk)
        i += chunk_size - overlap
    return chunks

chunks = _simple_chunk(LONG_DOCUMENT)
print(f"Chunked into {len(chunks)} chunks")
print(f"Avg chunk size: {sum(len(c.split()) for c in chunks) // len(chunks)} words")
print(f"\nFirst chunk preview:")
print(" ".join(chunks[0].split()[:50]) + "...")


In [ ]:
def _retrieve(query: str, chunks: list, top_k: int = 3) -> list:
    """Simple keyword-overlap retrieval -- no vector DB needed."""
    query_words = set(query.lower().split())
    scored = []
    for chunk in chunks:
        chunk_words = set(chunk.lower().split())
        score = len(query_words & chunk_words)
        scored.append((score, chunk))
    scored.sort(key=lambda x: x[0], reverse=True)
    return [c for _, c in scored[:top_k] if _ > 0]

# Test on Q1
test_q = QUESTIONS[0]["q"]
retrieved = _retrieve(test_q, chunks)
print(f"Query: {test_q}")
print(f"Retrieved {len(retrieved)} chunks\n")
for i, c in enumerate(retrieved, 1):
    print(f"Chunk {i}: {" ".join(c.split()[:40])}...\n")


In [ ]:
def answer_with_chunked_rag(document: str, questions: list) -> list:
    """Answer questions using keyword-overlap chunked retrieval."""
    client = ChatOpenAI(model=MODEL, temperature=0)
    doc_chunks = _simple_chunk(document)
    answers = []
    for item in questions:
        retrieved = _retrieve(item["q"], doc_chunks)
        if not retrieved:
            answers.append("No relevant chunks found.")
            continue
        context = "\n\n---\n\n".join(retrieved)
        system = "Answer using ONLY the context below. Be brief.\n\nCONTEXT:\n" + context
        msg = [SystemMessage(content=system), HumanMessage(content=item["q"])]
        response = client.invoke(msg)
        answers.append(response.content.strip())
    return answers

print("answer_with_chunked_rag defined")


In [ ]:
print("Running chunked RAG approach (all 5 questions)...\n")
t1 = time.time()
rag_answers = answer_with_chunked_rag(LONG_DOCUMENT, QUESTIONS)
rag_latency = time.time() - t1

for i, (q, a) in enumerate(zip(QUESTIONS, rag_answers), 1):
    print(f"Q{i}: {q['q'][:60]}...")
    print(f"  Expected: {q['expected']}")
    print(f"  Got:      {a[:100]}")
    print()

print(f"Total latency: {rag_latency:.1f}s ({rag_latency/len(QUESTIONS):.1f}s per question)")


## Part 6 — Scoring and Comparison

We use a simple partial-match scorer:
- **1.0** — expected answer substring found in the model output
- **partial** — overlap of expected keywords found
- **0.0** — no overlap

In production you would use an LLM-as-judge or annotated ground truth.


In [ ]:
def score_answer(predicted: str, expected: str) -> float:
    """Partial-match score: 1.0 if expected in predicted, else keyword overlap ratio."""
    if expected.lower() in predicted.lower():
        return 1.0
    exp_words = set(expected.lower().replace("%", "").split())
    pred_words = set(predicted.lower().replace("%", "").split())
    overlap = len(exp_words & pred_words)
    return round(overlap / max(len(exp_words), 1), 2)

results = []
for i, q in enumerate(QUESTIONS):
    fs = score_answer(full_answers[i], q["expected"])
    rs = score_answer(rag_answers[i], q["expected"])
    results.append({"q": i+1, "expected": q["expected"],
                    "full_score": fs, "rag_score": rs,
                    "winner": "full" if fs >= rs else "rag"})

print(f"{'Q':<3} {'Expected':<22} {'Full':>6} {'RAG':>6} {'Winner':>8}")
print("-" * 52)
for r in results:
    print(f"{r['q']:<3} {r['expected']:<22} {r['full_score']:>6.2f} {r['rag_score']:>6.2f} {r['winner']:>8}")

avg_full = sum(r["full_score"] for r in results) / len(results)
avg_rag = sum(r["rag_score"] for r in results) / len(results)
print(f"\nAverage{'':<18} {avg_full:>6.2f} {avg_rag:>6.2f}")


## Part 7 — Latency and Cost

From the report:

| Metric | Full Context | Chunked RAG |
|--------|-------------|-------------|
| Faithfulness | 0.91 | 0.83 |
| Latency p50 | 4.2s | 1.8s |
| Cost/1K queries | $0.84 | $0.22 |
| Multi-hop accuracy | 89% | 67% |

Full-context is **2.3x slower** and **3.8x more expensive** — but meaningfully
more accurate for multi-hop reasoning.


In [ ]:
print("=== Observed Latency ===")
print(f"Full context:  {full_latency:.1f}s total  ({full_latency/len(QUESTIONS):.1f}s per question)")
print(f"Chunked RAG:   {rag_latency:.1f}s total  ({rag_latency/len(QUESTIONS):.1f}s per question)")
speedup = full_latency / max(rag_latency, 0.001)
print(f"Speedup:       RAG is {speedup:.1f}x faster than full context")

print("\n=== Accuracy Summary ===")
print(f"Full context avg score: {avg_full:.2f}")
print(f"Chunked RAG avg score:  {avg_rag:.2f}")
print(f"Accuracy gap:           {avg_full - avg_rag:+.2f} (positive = full wins)")
print(f"Full-context wins: {avg_full >= avg_rag}")


## Part 8 — Cost Analysis

### True cost per correct answer

Raw cost per 1K queries is misleading without adjusting for accuracy:

```
Full context:  $0.84 / 1K queries  / 0.91 faithfulness = $0.92 / correct answer
Chunked RAG:   $0.22 / 1K queries  / 0.83 faithfulness = $0.27 / correct answer
```

At 1M queries/month: **$650K annual difference**.

### Break-even rule

| Faithfulness needed | Best approach |
|--------------------|---------------|
| > 88% | Full context |
| 80-88% | Chunked RAG (top-5) |
| < 80% | No context (pure LLM) |

For legal, medical, and financial applications: full-context is cost-justified.


## Part 9 — Decision Framework

| Condition | Recommended Approach |
|-----------|---------------------|
| Document < 50K tokens AND multi-hop | Full context |
| Faithfulness > 88% required | Full context |
| Corpus > 100K tokens | Chunked RAG |
| Single-hop fact lookup | Chunked RAG |
| Latency < 2s required | Chunked RAG |
| Large corpus, concentrated queries | Hybrid (RAG + rerank + full context) |

**Key takeaway:** production systems tier their approach:
fast RAG for volume, full-context for precision, human escalation for edge cases.


## Exercises

Try these modifications to deepen your understanding.


### Exercise 1 — Smaller Chunks

Change `chunk_size` from `400` to `256` and re-run the chunked RAG approach.

Predict: will accuracy go up or down? Why?

The report says: *"Chunks below 256 tokens showed 23% higher hallucination rates."*

Hint: smaller chunks lose sentence-level context but may improve retrieval precision.


In [ ]:
# Exercise 1: change chunk_size to 256

def answer_with_small_chunks(document, questions, chunk_size=256, overlap=25):
    client = ChatOpenAI(model=MODEL, temperature=0)
    doc_chunks = _simple_chunk(document, chunk_size=chunk_size, overlap=overlap)
    print(f"Chunks with size={chunk_size}: {len(doc_chunks)} chunks")
    answers = []
    for item in questions:
        retrieved = _retrieve(item["q"], doc_chunks)
        if not retrieved:
            answers.append("No relevant chunks found.")
            continue
        context = "\n\n---\n\n".join(retrieved)
        system = "Answer using ONLY the context below. Be brief.\n\nCONTEXT:\n" + context
        msg = [SystemMessage(content=system), HumanMessage(content=item["q"])]
        response = client.invoke(msg)
        answers.append(response.content.strip())
    return answers

# Uncomment to run:
# small_answers = answer_with_small_chunks(LONG_DOCUMENT, QUESTIONS)
# for i, (q, a) in enumerate(zip(QUESTIONS, small_answers), 1):
#     s = score_answer(a, q["expected"])
#     print(f"Q{i}: score={s:.2f}  got: {a[:80]}")


### Exercise 2 — Add a Sixth Question

Add a new multi-hop question combining facts from two sections.
Run it through both approaches and compare scores.

Suggested question:
*"What percentage of organizations use the most-deployed embedding model,
and what is its NDCG@5 retrieval accuracy?"*

Expected: *"89% use text-embedding-3-small, NDCG@5 = 0.847"*


In [ ]:
# Exercise 2: add a sixth question
Q6 = {
    "q": "What percentage of organizations use the most-deployed embedding model, and what is its NDCG@5 retrieval accuracy?",
    "expected": "89% use text-embedding-3-small, NDCG@5 = 0.847",
}

# Uncomment to run:
# full_q6 = answer_with_full_context(LONG_DOCUMENT, [Q6])
# rag_q6 = answer_with_chunked_rag(LONG_DOCUMENT, [Q6])
# print(f"Full: {full_q6[0]}")
# print(f"RAG:  {rag_q6[0]}")
# print(f"Full score: {score_answer(full_q6[0], Q6["expected"])}")
# print(f"RAG  score: {score_answer(rag_q6[0], Q6["expected"])}")


## Answer Key

### Exercise 1 — Expected outcome

With `chunk_size=256`, accuracy typically **decreases** because:
- Sentences get split mid-thought, reducing coherence
- The 23% higher hallucination rate from the report is a real effect
- Smaller chunks have less context for multi-hop reasoning

### Exercise 2 — Expected outcome

This question requires:
- Hop 1: Section 4.2 → "text-embedding-3-small: 89% of deployments"
- Hop 2: Section 4.2 → "NDCG@5: 0.847"

Both facts are in the same section, so chunked RAG should perform well.
The gap between full-context and RAG will be **smaller** than for questions
requiring facts from Sections 1 and 3 simultaneously.


## Workshop Complete

You have learned:

1. **Lost-in-the-middle**: models struggle with content in the middle of long contexts
2. **Full-context**: more faithful (0.91 vs 0.83), 2.3x slower, 3.8x costlier
3. **Chunked RAG**: faster and cheaper, 20-25% accuracy gap on multi-hop questions
4. **Partial-match scoring**: a lightweight proxy for faithfulness
5. **Decision rule**: faithfulness > 88% → full context; otherwise → RAG

**Next steps:**
- Replace keyword retrieval with vector embeddings (`text-embedding-3-small`)
- Try hybrid: RAG to find candidates + full-context on selected chunks
- Use an LLM-as-judge for production-grade faithfulness scoring

**Reference:** Liu et al. (2023) — [arxiv:2307.03172](https://arxiv.org/abs/2307.03172)
